# Metoda Crank-Nicholson 
- dla Równania Schrodingera zależnego od czasuw 1D

In [ ]:
# Blokada wielowątkowości
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

In [ ]:
# Biblioteki
import pickle
import numpy as np
from time import perf_counter
from IPython.display import HTML
import matplotlib.pyplot as plt
from scipy.sparse import diags, eye
from scipy.sparse.linalg import spsolve
from matplotlib.animation import FuncAnimation
from matplotlib_inline.backend_inline import set_matplotlib_formats
set_matplotlib_formats('svg')

In [ ]:
# Parametry
h_bar = 1.0
m = 1.0

In [ ]:
# 1 Funkcja do potencjałów
def V_func(_x, _N, _L, _xb, _Vb, _gamma_on, _xg, _xa, _n):
    V = np.zeros(_N, dtype=complex)
    for i in range(_N):
        if _x[i] > _xb and _x[i] < (_xb + _L/10): # Bariera
            V[i] = _Vb
        if _gamma_on and _x[i] > _xg: # Wchłaniacz
            V[i] -= 1j * _xa * ((_x[i] - _xg) / (_L - _xg))**_n            
    return V

In [ ]:
# 2 Funkcja do pakietu Gaussa
def Gauss(_psi, _x, _x0, _sigma, _k0):
    _psi[:] = np.exp(-((_x - _x0) / (2.0*_sigma))**2) * np.exp(1j *_k0*(_x - _x0))

In [ ]:
# 3 Funkcja do normalizacji funkcji falowej
def Norm(_psi, _dx):
    norm = np.sqrt(np.sum(np.abs(_psi)**2) * _dx)
    _psi[:] = _psi / norm

In [ ]:
# 4 Funkcja do inicjalizacji
def initialize(T_max, N, L, dt, x0, sigma, k0, xb0, Vb, gamma_on, xg0, xa, n):
    # Położenie, czas i stała przeskoku
    dx = L / (N - 1)
    print(f"dx = {dx:.3f} \nsigma = {sigma}")
    nt = int(T_max / dt)
    x = np.linspace(0, L, N)
    t = h_bar**2 / (2 * m * dx**2)
    # Energia
    E0 = (h_bar*k0)**2 / (2.0*m)
    print(f'Energia = {E0:.3f} Bariera = {Vb}')
    # Potencjał
    V = V_func(x, N, L, xb0, Vb, gamma_on, xg0, xa, n)
    # Hamiltonian
    side_diag = -t * np.ones(N - 1)
    main_diag = 2 * t + V
    H = diags([side_diag, main_diag, side_diag], [-1, 0, 1], format='csr')
    # Macierze A i B
    A = eye(N, format='csr') + (1j * dt / (2 * h_bar)) * H
    B = eye(N, format='csr') - (1j * dt / (2 * h_bar)) * H
    # Inicjalizacja f-cji falowej
    psi = np.zeros(N, dtype=complex)
    Gauss(_psi=psi, _x=x, _x0=x0, _sigma=sigma, _k0=k0)
    Norm(psi, dx)
    return psi, V, A, B, x, dx, nt, L

In [ ]:
# 5 Funkcja do rozwiązania Crank-Nicholsonem
def solve(filename, psi, A, B, x, dx, dt, nt, t1, t2, t3):
    results = []
    # Wybrane chwile na 3 wykresach
    time_plot = [t1, t2, t3] 
    static_plots = []
    for i in range(nt):
        # Crank-Nicolson + Normalizacja
        psi = spsolve(A, B.dot(psi))
        Norm(psi, dx)
        # Gęstość prawdopodobieństwa
        rho = np.abs(psi)**2
        # Położenie oczekiwane
        x_avg = np.sum(x * rho) * dx
        results.append([i*dt, rho, x_avg])
        if (i == time_plot[0] or i == time_plot[1] or i == time_plot[2]): 
            static_plots.append((i*dt, rho.copy()))
    with open(f"{filename}.pkl", "wb") as f: 
        pickle.dump(results, f)
    return static_plots, f"{filename}.pkl"

In [ ]:
# 6 Funkcja do wykresów 3 chwil czasowych
def plot_static(data, s, ymin, ymax, _L):
    plt.figure(figsize=(s, 3*s/4))
    for i in range(len(static)):
        plt.subplot(3, 1, i + 1)
        t_val = static[i][0]
        rho_val = static[i][1]
        plt.plot(X, V.real, 'r-', alpha=0.2)
        plt.fill_between(X, 0, V.real, color='r', alpha=0.05)
        plt.plot(X, rho_val, 'b-')
        plt.title(f"t = {t_val}")
        plt.xlim(0,_L); plt.ylim(ymin, ymax); plt.grid(True)
        
    plt.show()

In [ ]:
# 7 Funkcja do animacji (Ja tylko ją modyfikowałem - Gemini napisał)
def gif(file, step, _interval, s, ymin, ymax, _L):
    with open(file, "rb") as f: data = pickle.load(f)
    _frames = len(data)//step
    plt.figure(figsize=(2*s, 3*s/4))
    line, = plt.plot([], [], 'b-', lw=1.5)
    plt.plot(X, V.real, 'r-', alpha=0.3)
    plt.fill_between(X, 0, V.real, color='r', alpha=0.05)
    plt.ylim(ymin, ymax); plt.xlim(0, _L); plt.grid(True, alpha=0.2)

    def update(frame):
        i = frame * step
        line.set_data(X, data[i][1])
        return line
    anim = FuncAnimation(plt.gcf(), update, frames=_frames, interval=_interval)
    plt.close()
    return HTML(anim.to_jshtml())

In [ ]:
# 8 Funkcja do trajketorii całki położenia pakietu 
def trajectory(file, xb, xg, s):
    with open(file, "rb") as f: data = pickle.load(f)
    time_vec = [row[0] for row in data]
    x_avg_vec = [row[2] for row in data]
    plt.figure(figsize=(2*s, s))
    plt.plot(time_vec, x_avg_vec, 'g-', lw=2, label=r'$\langle x \rangle(t)$')
    if(xb): plt.axhline(xb, color='r', ls='--', alpha=0.3, label='Bariera')
    if(xg): plt.axhline(xg, color='b', ls='--', alpha=0.3, label='Wygaszacz')
    plt.xlabel('Czas t'); plt.ylabel('Srednie polozenie <x>')
    plt.title('Ewolucja wartosci oczekiwanej polozenia')
    plt.grid(True, alpha=0.2); plt.legend(); plt.show()

# 1.1. Bez żadnego potencjału $\sigma_0 > dx$:

In [ ]:
Psi, V, A, B, X, dx, Nt, L = initialize(T_max=80.0, N=600, L=100.0, dt=0.05, 
                                     x0=10.0, sigma=1.5, k0=4, 
                                     xb0=45, Vb=0, 
                                        gamma_on=False, xg0=80, xa=10, n=3)
static, file1 = solve('out1_04', Psi, A, B, X, dx, 0.05, Nt, t1=0, t2=200, t3=400)
plot_static(data=static, s=7, ymin=-0.05, ymax=0.3, _L=L)
trajectory(file=file1, xb=False, xg=False, s=4)
gif(file=file1, step=20, _interval=50, s=7, ymin=-0.05, ymax=0.3, _L=L)

# Komentarz ad.1.1
- Swobodna cząstka zadana funkcją Gaussa w pudle w podejściu falowym porusza się ruchem jednostajnym, 
co obrazuje liniowy wykres trajektorii ⟨x⟩(t). 
- Wraz z upływem czasu widoczne jest wyraźne rozmycie pakietu, odbijanie od "ścian" pudła oraz zachowanie normy prawdopodobieństwa dzięki unitarności schematu Crank-Nicolsona.
- Po testach bez normalizacji zarówno w inicjalizacji pakietu Gaussowskiego jak i bez inicjalizacji w kroku czasowym oraz i tu i tu, nie zaobserwowałem różnicy w trajektorii co świadczy również o stabilbości i dokładności tej metody w ewolucji czasowej o takiej długości.

(Oczywiście bez normalizacji w inicjalizacji pakiet miał dużo większą gęstość stanu, ale w interpretacji gęstości stanu jako gęst. prawdopodoebieństwa jest to nie poprawne.)

# 1.2. Bez żadnego potencjału $\sigma_0 < dx$:

In [ ]:
Psi, V, A, B, X, dx, Nt, L = initialize(T_max=80.0, N=600, L=100.0, dt=0.05, 
                                     x0=10.0, sigma=0.1, k0=1, 
                                     xb0=45, Vb=0, 
                                        gamma_on=False, xg0=80, xa=10, n=3)
static, file12 = solve('out12_04', Psi, A, B, X, dx, 0.05, Nt, t1=0, t2=200, t3=400)
plot_static(data=static, s=7, ymin=-0.05, ymax=0.3, _L=L)
trajectory(file=file12, xb=False, xg=False, s=4)
gif(file=file12, step=20, _interval=50, s=7, ymin=-0.05, ymax=0.3, _L=L)

# Komentarz ad.1.2
- Przy ekstremalnie małej wartości $\sigma_0$ pakiet gwałtownie "rozsypuje się" na skutek ogromnej nieoznaczoności pędu $\Delta p \approx 1/\sigma$, co objawia się m.in. nieliniowym przebiegiem trajektorii $⟨x⟩(t)$.
- Główną przyczyną błędu jest niewystarczająca rozdzielczość siatki $dx \approx 0.16$, która nie jest w stanie odwzorować funkcji zmieniającej się na dystansie $\sigma < dx $, co generuje niefizyczne oscylacje numeryczne i zaburza działanie operatora ewolucji.
- Wynik ten stanowi dowód na istnienie ograniczeń metody Cranka-Nicolsona przy zadanych parametrach dyskretyzacji i pokazuje, że zbyt silna lokalizacja przestrzenna wymaga gęstszej siatki (większego N) dla zachowania stabilności i sensu fizycznego wyników.

# 2. Bariera potencjału $V_0$ < E:

In [ ]:
Psi, V, A, B, X, dx, Nt, L = initialize(T_max=80.0, N=600, L=100.0, dt=0.05, 
                                     x0=10.0, sigma=1.5, k0=4, 
                                     xb0=45, Vb=5, 
                                        gamma_on=False, xg0=80, xa=10, n=3)
static, file2 = solve('out2_04', Psi, A, B, X, dx, 0.05, Nt, t1=0, t2=240, t3=400)
plot_static(data=static, s=7, ymin=-0.05, ymax=0.3, _L=L)
trajectory(file=file2, xb=45, xg=False, s=4)
gif(file=file2, step=20, _interval=50, s=7, ymin=-0.05, ymax=0.4, _L=L)

# Komentarz ad.2
- Przy kontakcie z barierą $V_b$ = 5.0 następuje rozdzielenie pakietu na część odbitą i przetransmitowaną, co skutkuje gwałtownym załamaniem trajektorii $⟨x⟩(t)$ i zmianą jej nachylenia.
- Zjawisko to indukuje silne oscylacje interferencyjne wew. bariery, a środek masy układu przestaje poruszać się liniowo, odzwierciedlając podział gęstości prawdopodobieństwa na dwa oddalające się od siebie maksima.
- Schemat pozostaje stabilny i ściśle zachowuje normę.

# 3. Bariera potencjału $V_0$ > E:

In [ ]:
Psi, V, A, B, X, dx, Nt, L = initialize(T_max=80.0, N=600, L=100.0, dt=0.05, 
                                     x0=10.0, sigma=1.5, k0=4, 
                                     xb0=45, Vb=9, 
                                        gamma_on=False, xg0=80, xa=10, n=3)
static, file3 = solve('out3_04', Psi, A, B, X, dx, 0.05, Nt, t1=0, t2=240, t3=550)
plot_static(data=static, s=7, ymin=-0.05, ymax=0.3, _L=L)
trajectory(file=file3, xb=45, xg=False, s=4)
gif(file=file3, step=20, _interval=50, s=7, ymin=-0.05, ymax=0.4, _L=L)

# Komentarz ad.3
- Przy kontakcie z barierą $V_b$ = 9.0 następuje rozdzielenie pakietu na część odbitą i przetransmitowaną (tunelowanie), co skutkuje gwałtownym załamaniem trajektorii $⟨x⟩(t)$ i zmianą jej nachylenia.
- Zjawisko to indukuje silne oscylacje interferencyjne przed barierą, ale z uwagi na bardzo małą tunelującą "część" cząstki środek masy układu dalej porusza się liniowo.
- Schemat pozostaje stabilny i ściśle zachowuje normę..

# 4. Potencjał urojony "wygaszacz":

In [ ]:
Psi, V, A, B, X, dx, Nt, L = initialize(T_max=100.0, N=300, L=100.0, dt=0.1, 
                                     x0=10.0, sigma=1.5, k0=4, 
                                     xb0=45, Vb=0, 
                                        gamma_on=True, xg0=80, xa=10, n=4)
static, file4 = solve('out4_04', Psi, A, B, X, dx, 0.05, Nt, t1=0, t2=240, t3=550)
plot_static(data=static, s=7, ymin=-0.05, ymax=0.3, _L=L)
trajectory(file=file4, xb=False, xg=80, s=4)
gif(file=file4, step=20, _interval=50, s=7, ymin=-0.05, ymax=0.4, _L=L)

# Komentarz ad.4
- Wprowadzenie zespolonego potencjału pochłaniającego $\Gamma$ skutkuje łamaniem unitarności ewolucji, co objawia się wykładniczym spadkiem normy prawdopodobieństwa w momencie wejścia pakietu w obszar $x>80$.
- Trajektoria $⟨x⟩(t)$ rośnie liniowo do momentu kawałek za granicą pochłaniacza, 
chwile się tam utrzymuje po czym "przeskakuje" z powrotem przed pochałaniacz i tam się kończy życie cząstki.
- Zastosowanie gładkiego profilu potęgowego (n=4) pozwala na skuteczne wygaszenie fali bez generowania sztucznych odbić numerycznych od brzegów układu, co potwierdza poprawność doboru parametrów, jednak dla $n>8$ "fala" fala odbija się niczym od ściany zostawiając swój kawałek za progiem potencjału.